In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp05/ex_6.py ──
"""
Shared infrastructure for Exercise 6 — Graph Neural Networks.

Contains: Cora dataset loading (with Karate Club fallback), graph
normalisation, train/val/test split, kailash-ml engine setup,
graph visualisation helpers, and training harness.

Technique-specific code (GCN, GAT, GraphSAGE layers) does NOT belong here.
"""

import asyncio
import pickle
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker, ModelVisualizer
from kailash_ml import ModelRegistry
from kailash_ml.types import MetricSpec


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()

OUTPUT_DIR = Path("outputs") / "ex6_gnns"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "cora"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# GRAPH DATA LOADING
# ════════════════════════════════════════════════════════════════════════


def load_cora() -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, str, int]:
    """Cora — 2708 papers, 1433 bag-of-words features, 7 classes.

    Returns:
        X_np: node features (N, F)
        A_np: dense adjacency matrix (N, N)
        y_np: node labels (N,)
        edge_index_np: edge list (2, E) for link prediction
        dataset_name: "Cora"
        n_classes: 7
    """
    from torch_geometric.datasets import Planetoid

    dataset = Planetoid(root=str(DATA_DIR), name="Cora")
    # torch_geometric.data.Data has dynamic attributes (num_nodes/x/y/edge_index);
    # use Any to avoid type-checker false positives without losing runtime fidelity.
    data: Any = dataset[0]
    n = data.num_nodes
    X_np = data.x.numpy().astype(np.float32)
    y_np = data.y.numpy().astype(np.int64)

    # Build a dense adjacency matrix from the edge_index. Cora has ~10k
    # directed edges (5278 undirected) over 2708 nodes; the dense matrix
    # is ~7M entries which fits comfortably in CPU memory.
    A_np = np.zeros((n, n), dtype=np.float32)
    edge_index_np = data.edge_index.numpy()
    src = edge_index_np[0]
    dst = edge_index_np[1]
    A_np[src, dst] = 1.0
    A_np[dst, src] = 1.0  # symmetrise just in case
    n_classes = int(dataset.num_classes)
    return X_np, A_np, y_np, edge_index_np, "Cora", n_classes


def load_karate() -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, str, int]:
    """Zachary's Karate Club — 34 nodes, 78 edges, 2 factions."""
    import networkx as nx

    G = nx.karate_club_graph()
    n = G.number_of_nodes()
    A_np = nx.to_numpy_array(G, dtype=np.float32)
    labels = np.array(
        [0 if G.nodes[i]["club"] == "Mr. Hi" else 1 for i in range(n)],
        dtype=np.int64,
    )
    # Karate has no node features; use one-hot identity (transductive)
    X_np = np.eye(n, dtype=np.float32)
    # Build edge_index from adjacency
    src, dst = np.where(A_np > 0)
    edge_index_np = np.stack([src, dst]).astype(np.int64)
    return X_np, A_np, labels, edge_index_np, "Karate Club", 2


def load_graph_data() -> dict:
    """Load Cora (with Karate fallback) and return all graph tensors.

    Returns a dict with keys:
        X, A, y, A_norm, A_hat — torch tensors on device
        X_np, A_np, y_np, edge_index_np — numpy arrays
        train_mask, val_mask, test_mask — boolean masks on device
        N, F_dim, n_classes, n_edges_undirected, dataset_name — scalars
    """
    try:
        X_np, A_np, y_np, edge_index_np, dataset_name, n_classes = load_cora()
    except Exception as exc:
        print(
            f"Could not load Cora ({type(exc).__name__}: {exc}); "
            "falling back to Karate Club"
        )
        X_np, A_np, y_np, edge_index_np, dataset_name, n_classes = load_karate()

    N = X_np.shape[0]
    F_dim = X_np.shape[1]
    n_edges_undirected = int(A_np.sum() // 2)
    print(
        f"Graph: {dataset_name} — {N} nodes, {n_edges_undirected} undirected edges, "
        f"feature_dim={F_dim}, classes={n_classes}"
    )
    class_counts = ", ".join(f"c{c}={int((y_np == c).sum())}" for c in range(n_classes))
    print(f"  per-class counts: {class_counts}")

    X = torch.from_numpy(X_np).to(device)
    A = torch.from_numpy(A_np).to(device)
    y = torch.from_numpy(y_np).to(device)

    # Add self-loops and build the symmetric Laplacian D^{-1/2} A D^{-1/2}
    A_hat = A + torch.eye(N, device=device)
    deg = A_hat.sum(dim=1)
    d_inv_sqrt = deg.pow(-0.5)
    d_inv_sqrt[torch.isinf(d_inv_sqrt)] = 0.0
    A_norm = d_inv_sqrt.unsqueeze(1) * A_hat * d_inv_sqrt.unsqueeze(0)

    # Train/val/test split — 20% train, 20% val, 60% test (per class)
    train_mask = torch.zeros(N, dtype=torch.bool, device=device)
    val_mask = torch.zeros(N, dtype=torch.bool, device=device)
    rng = np.random.default_rng(0)
    for c in range(n_classes):
        idx = np.where(y_np == c)[0]
        if len(idx) == 0:
            continue
        rng.shuffle(idx)
        n_train = max(1, int(0.2 * len(idx)))
        n_val = max(1, int(0.2 * len(idx)))
        train_mask[idx[:n_train]] = True
        val_mask[idx[n_train : n_train + n_val]] = True
    test_mask = ~(train_mask | val_mask)
    print(
        f"  train: {int(train_mask.sum().item())}, "
        f"val: {int(val_mask.sum().item())}, "
        f"test: {int(test_mask.sum().item())}"
    )

    return {
        "X": X,
        "A": A,
        "y": y,
        "A_norm": A_norm,
        "A_hat": A_hat,
        "X_np": X_np,
        "A_np": A_np,
        "y_np": y_np,
        "edge_index_np": edge_index_np,
        "train_mask": train_mask,
        "val_mask": val_mask,
        "test_mask": test_mask,
        "N": N,
        "F_dim": F_dim,
        "n_classes": n_classes,
        "n_edges_undirected": n_edges_undirected,
        "dataset_name": dataset_name,
    }


# ════════════════════════════════════════════════════════════════════════
# KAILASH ENGINE SETUP
# ════════════════════════════════════════════════════════════════════════


async def _setup_engines():
    """Open kailash-ml 1.1.1 tracker + registry. 5-tuple preserved."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_gnns.db"
    registry_db = "sqlite:///mlfp05_gnns_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return conn, tracker, "m5_gnns", registry, True


def setup_engines() -> tuple:
    """Synchronously set up kailash-ml engines."""
    return asyncio.run(_setup_engines())


# ════════════════════════════════════════════════════════════════════════
# TRAINING HARNESS
# ════════════════════════════════════════════════════════════════════════


def train_node_classifier(
    model: nn.Module,
    name: str,
    forward_arg: torch.Tensor,
    graph_data: dict,
    tracker: ExperimentTracker,
    exp_name: str,
    epochs: int = 100,
    lr: float = 1e-2,
    weight_decay: float = 5e-4,
) -> tuple[list[float], list[float], list[float]]:
    """Train a GNN for node classification and log metrics to ExperimentTracker.

    Returns:
        train_losses: per-epoch training loss
        val_accs: per-epoch validation accuracy
        test_accs: per-epoch test accuracy
    """
    return asyncio.run(
        _train_node_classifier_async(
            model,
            name,
            forward_arg,
            graph_data,
            tracker,
            exp_name,
            epochs,
            lr,
            weight_decay,
        )
    )


async def _train_node_classifier_async(
    model: nn.Module,
    name: str,
    forward_arg: torch.Tensor,
    graph_data: dict,
    tracker: ExperimentTracker,
    exp_name: str,
    epochs: int = 100,
    lr: float = 1e-2,
    weight_decay: float = 5e-4,
) -> tuple[list[float], list[float], list[float]]:
    """Async core — uses the kailash-ml 1.1.1 ``tracker.track(...)`` context manager."""
    X = graph_data["X"]
    y = graph_data["y"]
    train_mask = graph_data["train_mask"]
    val_mask = graph_data["val_mask"]
    test_mask = graph_data["test_mask"]
    N = graph_data["N"]
    n_edges = graph_data["n_edges_undirected"]
    dataset_name = graph_data["dataset_name"]
    hidden_dim = 16 if dataset_name == "Karate Club" else 64

    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  [{name}] parameters: {n_params:,}")

    train_losses: list[float] = []
    val_accs: list[float] = []
    test_accs: list[float] = []

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(
            {
                "model_type": name,
                "hidden_dim": str(hidden_dim),
                "epochs": str(epochs),
                "lr": str(lr),
                "weight_decay": str(weight_decay),
                "n_params": str(n_params),
                "dataset": dataset_name,
                "n_nodes": str(N),
                "n_edges": str(n_edges),
            }
        )

        for epoch in range(epochs):
            model.train()
            opt.zero_grad()
            logits = model(X, forward_arg)
            loss = F.cross_entropy(logits[train_mask], y[train_mask])
            loss.backward()
            opt.step()
            train_losses.append(loss.item())

            model.eval()
            with torch.no_grad():
                preds = model(X, forward_arg).argmax(dim=-1)
                v_acc = (preds[val_mask] == y[val_mask]).float().mean().item()
                t_acc = (preds[test_mask] == y[test_mask]).float().mean().item()
            val_accs.append(v_acc)
            test_accs.append(t_acc)

            await run.log_metrics(
                {
                    "train_loss": loss.item(),
                    "val_accuracy": v_acc,
                    "test_accuracy": t_acc,
                },
                step=epoch + 1,
            )

            if (epoch + 1) % 25 == 0:
                print(
                    f"  [{name}] epoch {epoch+1:3d}  "
                    f"loss={loss.item():.4f}  val_acc={v_acc:.3f}  test_acc={t_acc:.3f}"
                )

        await run.log_metrics(
            {
                "final_loss": train_losses[-1],
                "final_val_accuracy": val_accs[-1],
                "final_test_accuracy": test_accs[-1],
                "best_val_accuracy": max(val_accs),
                "best_test_accuracy": max(test_accs),
            }
        )

    return train_losses, val_accs, test_accs


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION HELPERS
# ════════════════════════════════════════════════════════════════════════

viz = ModelVisualizer()


def plot_training_curves(
    metrics_dict: dict[str, list[float]],
    title: str,
    y_label: str,
    filename: str,
) -> None:
    """Plot overlaid training curves and save as HTML."""
    fig = viz.training_history(
        metrics=metrics_dict,
        x_label="Epoch",
        y_label=y_label,
    )
    fig.update_layout(title=title)
    filepath = OUTPUT_DIR / filename
    fig.write_html(str(filepath))
    print(f"  Saved: {filepath}")


def plot_node_embeddings(
    embeddings: np.ndarray,
    labels: np.ndarray,
    n_classes: int,
    title: str,
    filename: str,
) -> None:
    """2-D PCA projection of node embeddings coloured by class label.

    Uses SVD-based PCA (no sklearn dependency). Nodes of the same class
    should cluster together if the GNN learned meaningful representations.
    """
    emb_centered = embeddings - embeddings.mean(axis=0, keepdims=True)
    Vt = np.linalg.svd(emb_centered, full_matrices=False)[2]
    coords = emb_centered @ Vt.T[:, :2]

    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    cmap = plt.cm.get_cmap("tab10", n_classes)
    for c in range(n_classes):
        mask = labels == c
        ax.scatter(
            coords[mask, 0],
            coords[mask, 1],
            c=[cmap(c)],
            s=15,
            alpha=0.6,
            label=f"Class {c}",
        )
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("PC 1")
    ax.set_ylabel("PC 2")
    ax.legend(fontsize=8, markerscale=2, loc="best")
    plt.tight_layout()
    filepath = OUTPUT_DIR / filename
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {filepath}")

    # Text summary of first 3 nodes per class
    print(f"\n  {title} — first 3 nodes per class:")
    for c in range(min(n_classes, 7)):
        rows = coords[labels == c][:3]
        if len(rows) == 0:
            continue
        pretty = ", ".join(f"({r[0]:+.2f}, {r[1]:+.2f})" for r in rows)
        print(f"    class {c}: {pretty}")

    return coords


def plot_graph_with_embeddings(
    A_np: np.ndarray,
    embeddings_2d: np.ndarray,
    labels: np.ndarray,
    n_classes: int,
    title: str,
    filename: str,
    max_nodes: int = 200,
) -> None:
    """Draw graph edges on the 2-D embedding space, coloured by class.

    Subsamples to max_nodes for readability on large graphs.
    """
    N = A_np.shape[0]
    if N > max_nodes:
        rng = np.random.default_rng(42)
        subset = rng.choice(N, max_nodes, replace=False)
    else:
        subset = np.arange(N)

    coords = embeddings_2d[subset]
    sub_labels = labels[subset]
    sub_A = A_np[np.ix_(subset, subset)]

    fig, ax = plt.subplots(1, 1, figsize=(12, 9))
    cmap = plt.cm.get_cmap("tab10", n_classes)

    # Draw edges first (behind nodes)
    src_idx, dst_idx = np.where(np.triu(sub_A) > 0)
    for s, d in zip(src_idx, dst_idx):
        ax.plot(
            [coords[s, 0], coords[d, 0]],
            [coords[s, 1], coords[d, 1]],
            color="gray",
            alpha=0.08,
            linewidth=0.5,
        )

    # Draw nodes
    for c in range(n_classes):
        mask = sub_labels == c
        ax.scatter(
            coords[mask, 0],
            coords[mask, 1],
            c=[cmap(c)],
            s=25,
            alpha=0.7,
            label=f"Class {c}",
            zorder=2,
        )

    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.legend(fontsize=8, markerscale=2, loc="best")
    ax.set_xlabel("PC 1")
    ax.set_ylabel("PC 2")
    plt.tight_layout()
    filepath = OUTPUT_DIR / filename
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {filepath}")


def plot_attention_weights(
    alpha: np.ndarray,
    A_np: np.ndarray,
    labels: np.ndarray,
    title: str,
    filename: str,
    node_idx: int = 0,
    top_k: int = 10,
) -> None:
    """Visualise attention weights for a single node's neighbourhood.

    Shows which neighbours the GAT layer attends to most strongly.
    """
    neighbours = np.where(A_np[node_idx] > 0)[0]
    if len(neighbours) == 0:
        print(f"  Node {node_idx} has no neighbours — skipping attention plot")
        return

    weights = alpha[node_idx, neighbours]
    order = np.argsort(-weights)[:top_k]
    top_neighbours = neighbours[order]
    top_weights = weights[order]

    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    bar_colors = [plt.cm.get_cmap("tab10")(labels[n] % 10) for n in top_neighbours]
    ax.barh(
        range(len(top_neighbours)),
        top_weights,
        color=bar_colors,
        edgecolor="white",
    )
    ax.set_yticks(range(len(top_neighbours)))
    ax.set_yticklabels(
        [f"Node {n} (class {labels[n]})" for n in top_neighbours],
        fontsize=9,
    )
    ax.set_xlabel("Attention Weight")
    ax.set_title(
        f"{title}\nNode {node_idx} (class {labels[node_idx]}) attending to top-{top_k} neighbours",
        fontsize=12,
        fontweight="bold",
    )
    ax.invert_yaxis()
    plt.tight_layout()
    filepath = OUTPUT_DIR / filename
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {filepath}")


# ════════════════════════════════════════════════════════════════════════
# MODEL REGISTRATION HELPER
# ════════════════════════════════════════════════════════════════════════


async def _register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    metrics: list[MetricSpec],
) -> object:
    """Register a model in the ModelRegistry."""
    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=name,
        artifact=model_bytes,
        metrics=metrics,
    )
    return version


def register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    metrics: list[MetricSpec],
) -> object:
    """Synchronously register a model."""
    return asyncio.run(_register_model(registry, name, model, metrics))




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 6.1: Graph Convolutional Network (GCN)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Why tabular ML methods fail on graph-structured data
#   - The message-passing intuition: nodes learn from their neighbours
#   - Build a GCN layer as torch.nn.Module:  H' = sigma(A_norm @ H @ W)
#   - Train a node classifier on the Cora citation network (2708 papers)
#   - Visualise learned node embeddings with 2-D PCA projection
#   - Track training with kailash-ml ExperimentTracker
#
# PREREQUISITES: M5/ex_4 (attention mechanisms, nn.Module training).
# ESTIMATED TIME: ~30 min
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from kailash_ml.types import MetricSpec



## PHASE 1 — THEORY: Why Graphs Need Their Own Neural Networks


WHY GRAPHS NEED SPECIAL ARCHITECTURES:
  - Tabular methods ignore relational structure (citation links)
  - Variable-size neighbourhoods can't be flattened to a fixed vector
  - Graph structure carries classification signal independent of features

  GCN MESSAGE PASSING (the "gossip protocol"):
  1. Collect — each node gathers features from neighbours
  2. Aggregate — average features (weighted by degree)
  3. Transform — pass through learnable weights
  4. Stack — multiple layers = multi-hop information flow

  Formula:  H' = sigma( D^{-1/2} A D^{-1/2} H W )
  The ENTIRE neighbourhood aggregation is ONE matrix multiply!


In [ ]:
#
# Imagine you're classifying research papers at NUS or NTU into fields
# like "Computer Vision", "NLP", or "Reinforcement Learning". You could
# use each paper's bag-of-words features in a standard MLP — but you'd
# be ignoring the most valuable signal: CITATION LINKS.
#
# A paper about "Attention Is All You Need" that cites 10 NLP papers and
# is cited by 50 more NLP papers is almost certainly an NLP paper — even
# if its bag-of-words features overlap with computer vision papers.
#
# The problem: tabular methods (MLP, logistic regression, random forest)
# expect a fixed-size feature vector per sample. But graph data has:
#   - Variable-size neighbourhoods (some papers cite 2, others cite 200)
#   - Relational structure (who-cites-whom) that carries class signal
#   - No natural ordering of neighbours
#
# GCN's insight: MESSAGE PASSING. Think of it like a gossip protocol:
#   1. Each node collects features from its neighbours
#   2. It averages them (weighted by degree)
#   3. It passes the average through a learnable transform
#   4. After 2 hops, each node "knows" about its 2-hop neighbourhood
#
# Mathematically:  H' = sigma( D^{-1/2} A D^{-1/2} H W )
#   - A = adjacency matrix (who's connected to whom)
#   - D = degree matrix (how many connections each node has)
#   - H = current node features
#   - W = learnable weight matrix
#   - sigma = activation function (ReLU)
#
# The key insight: the entire message-passing step is ONE MATRIX
# MULTIPLICATION. No Python loops over nodes. This is why GCNs are
# fast and parallelisable on GPUs.
print("=" * 70)
print("  PHASE 1 — THEORY: Graph Convolutional Networks")
print("=" * 70)
print(
)



## PHASE 2 — BUILD: GCN Layer Implementation


No Python loops over nodes — a single matmul aggregates every
    neighbourhood at once. This is the core insight of GCNs: message
    passing as matrix multiplication.


Two-layer GCN for node classification.


Return the hidden-layer embedding (before classification head).


In [ ]:
print("=" * 70)
print("  PHASE 2 — BUILD: GCN Layer + Model")
print("=" * 70)

# Load graph data and set up engines
graph_data = load_graph_data()
conn, tracker, exp_name, registry, has_registry = setup_engines()

X = graph_data["X"]
A = graph_data["A"]
A_norm = graph_data["A_norm"]
y = graph_data["y"]
y_np = graph_data["y_np"]
N = graph_data["N"]
F_dim = graph_data["F_dim"]
n_classes = graph_data["n_classes"]
dataset_name = graph_data["dataset_name"]

HIDDEN_DIM = 16 if dataset_name == "Karate Club" else 64
EPOCHS = 100


class GCNLayer(nn.Module):

    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        # TODO: Create a linear projection W that maps in_dim -> out_dim
        # Hint: nn.Linear(in_dim, out_dim, bias=True)
        pass

    def forward(self, h: torch.Tensor, a_norm: torch.Tensor) -> torch.Tensor:
        # TODO: GCN forward — project features through W, then multiply by A_norm
        # Hint: return a_norm @ self.W(h)
        # The matmul with A_norm aggregates each node's neighbourhood in one step
        pass


class GCN(nn.Module):

    def __init__(self, in_dim: int, hidden_dim: int, n_classes: int):
        super().__init__()
        # TODO: Create two GCN layers
        # Layer 1: in_dim -> hidden_dim
        # Layer 2: hidden_dim -> n_classes
        pass

    def forward(self, h: torch.Tensor, a_norm: torch.Tensor) -> torch.Tensor:
        # TODO: Two-layer forward pass with ReLU and dropout
        # 1. Pass through layer 1, apply ReLU
        # 2. Apply dropout (p=0.5) during training
        # 3. Pass through layer 2 (no activation — raw logits for cross-entropy)
        pass

    def embed(self, h: torch.Tensor, a_norm: torch.Tensor) -> torch.Tensor:
        # TODO: Return ReLU(layer1(h, a_norm)) — the intermediate representation
        pass


gcn = GCN(in_dim=F_dim, hidden_dim=HIDDEN_DIM, n_classes=n_classes)
n_params = sum(p.numel() for p in gcn.parameters())
print(f"\n  GCN architecture:")
print(f"    Layer 1: GCNLayer({F_dim} -> {HIDDEN_DIM})")
print(f"    Layer 2: GCNLayer({HIDDEN_DIM} -> {n_classes})")
print(f"    Total parameters: {n_params:,}")
print(f"\n  How it works:")
print(f"    Input:  {N} nodes x {F_dim} features")
print(f"    Layer 1: A_norm @ (X @ W1) -> {N} x {HIDDEN_DIM} (neighbourhood averages)")
print(f"    ReLU + Dropout(0.5)")
print(f"    Layer 2: A_norm @ (H @ W2) -> {N} x {n_classes} (class logits)")



### Build Checkpoint


In [ ]:
assert isinstance(gcn, nn.Module), "GCN should be an nn.Module"
assert n_params > 0, "GCN should have learnable parameters"
print("\n--- Build checkpoint passed --- GCN architecture created\n")



## PHASE 3 — TRAIN: Node Classification on Cora


In [ ]:
print("=" * 70)
print(f"  PHASE 3 — TRAIN: GCN on {dataset_name}")
print("=" * 70)

gcn_losses, gcn_val, gcn_test = train_node_classifier(
    model=gcn,
    name="GCN",
    forward_arg=A_norm,
    graph_data=graph_data,
    tracker=tracker,
    exp_name=exp_name,
    epochs=EPOCHS,
)



### Train Checkpoint


In [ ]:
assert len(gcn_losses) == EPOCHS, f"Expected {EPOCHS} epoch losses for GCN"
assert gcn_losses[-1] < gcn_losses[0], "GCN loss should decrease"
best_val = max(gcn_val)
best_test = max(gcn_test)
print(f"\n  GCN Results:")
print(f"    Best validation accuracy: {best_val:.4f}")
print(f"    Best test accuracy:       {best_test:.4f}")
print(f"    Final loss:               {gcn_losses[-1]:.4f}")
# INTERPRETATION: GCN uses a fixed aggregation scheme based on the graph
# Laplacian. Every node's new representation is a weighted average of its
# neighbours' features, where the weights come from the degree-normalised
# adjacency. This is equivalent to a 1-hop spectral filter on the graph.
print("\n--- Train checkpoint passed --- GCN trained successfully\n")



## PHASE 4 — VISUALISE: Node Embeddings + Graph Structure


In [ ]:
print("=" * 70)
print("  PHASE 4 — VISUALISE: GCN Node Embeddings")
print("=" * 70)

gcn.eval()
with torch.no_grad():
    gcn_emb = gcn.embed(X, A_norm).cpu().numpy()

# Plot 1: 2-D PCA of node embeddings coloured by class
coords = plot_node_embeddings(
    embeddings=gcn_emb,
    labels=y_np,
    n_classes=n_classes,
    title=f"GCN Node Embeddings — {dataset_name}",
    filename="gcn_node_embeddings.png",
)

# Plot 2: Graph structure overlaid on embedding space
plot_graph_with_embeddings(
    A_np=graph_data["A_np"],
    embeddings_2d=coords,
    labels=y_np,
    n_classes=n_classes,
    title=f"GCN — Graph Structure in Embedding Space ({dataset_name})",
    filename="gcn_graph_embeddings.png",
)

# Plot 3: Training curves
plot_training_curves(
    metrics_dict={"GCN train loss": gcn_losses},
    title="GCN Training Loss",
    y_label="Cross-Entropy Loss",
    filename="gcn_loss_curve.html",
)
plot_training_curves(
    metrics_dict={"GCN val accuracy": gcn_val, "GCN test accuracy": gcn_test},
    title="GCN Accuracy",
    y_label="Accuracy",
    filename="gcn_accuracy_curves.html",
)



### Visualise Checkpoint


In [ ]:
assert gcn_emb.shape == (
    N,
    HIDDEN_DIM,
), f"Embedding shape should be ({N}, {HIDDEN_DIM})"
# INTERPRETATION: Good GCN embeddings show clear class separation in the
# 2-D projection. Nodes of the same class cluster because the GCN
# aggregated features from their citation neighbourhoods — papers in the
# same field cite similar papers and share vocabulary, so their
# aggregated representations converge.
print("\n--- Visualise checkpoint passed --- GCN embeddings plotted\n")



## PHASE 5 — APPLY: Academic Research Network at NUS/NTU


SCENARIO: You're building a research analytics tool for NUS or NTU.
  The university publishes thousands of papers per year across faculties.
  Your task: automatically classify papers into research fields using
  both their text features AND citation links.

  WHY GCN BEATS BAG-OF-WORDS BASELINE:
  Consider a paper titled "Efficient Training of Vision Transformers".
  - Bag-of-words features: high weight on "training", "efficient", "vision"
    -> Could be Computer Vision OR Systems/Optimisation
  - Citation links: cites ViT, DeiT, DINO (all CV papers)
    -> GCN correctly classifies as Computer Vision

  The citation graph is a free signal that bag-of-words models ignore.


REAL-WORLD DEPLOYMENT:
  1. Build a citation graph from Scopus/Web of Science API
  2. Extract bag-of-words or TF-IDF features from abstracts
  3. Train GCN on papers with known faculty/department labels
  4. Classify new papers automatically for research analytics dashboards
  5. Track with ExperimentTracker — retrain quarterly as citation graph grows


In [ ]:
print("=" * 70)
print("  PHASE 5 — APPLY: Research Paper Classification at NUS/NTU")
print("=" * 70)
print(
)

# Demonstrate: compare a naive bag-of-words baseline to the GCN
print("  Bag-of-Words Baseline vs GCN Comparison:")

# TODO: Build a simple bag-of-words baseline (logistic regression on raw features)
# 1. Create a baseline model: nn.Linear(F_dim, n_classes).to(device)
# 2. Create an Adam optimizer with lr=1e-2
# 3. Train for EPOCHS: forward pass on X (no graph!), cross-entropy loss on train_mask
# 4. Evaluate: get predictions on test_mask, compute accuracy
# Hint: This is the same as GCN but WITHOUT the adjacency matrix multiplication
from torch.optim import Adam

train_mask = graph_data["train_mask"]
test_mask = graph_data["test_mask"]

# TODO: Create baseline_model and baseline_opt
# baseline_model = ...
# baseline_opt = ...

# TODO: Training loop — same as any classifier but uses X directly (no graph)
# for epoch in range(EPOCHS):
#     baseline_model.train()
#     baseline_opt.zero_grad()
#     logits = baseline_model(X)  # note: no A_norm!
#     loss = F.cross_entropy(logits[train_mask], y[train_mask])
#     loss.backward()
#     baseline_opt.step()

# TODO: Evaluate baseline
# baseline_model.eval()
# with torch.no_grad():
#     baseline_preds = baseline_model(X).argmax(dim=-1)
#     baseline_acc = (baseline_preds[test_mask] == y[test_mask]).float().mean().item()

baseline_acc = 0.0  # Replace with your computed accuracy

print(f"    Bag-of-Words (no graph):  test accuracy = {baseline_acc:.4f}")
print(f"    GCN (with graph):         test accuracy = {best_test:.4f}")
improvement = best_test - baseline_acc
print(f"    Improvement from graph:   +{improvement:.4f} ({improvement*100:.1f} pp)")
print()

if improvement > 0:
    print("  INSIGHT: The citation graph provides significant additional signal.")
    print("  Papers in the same field cite each other more often, creating")
    print("  class-homogeneous neighbourhoods that the GCN exploits.")
else:
    print("  NOTE: On this split, bag-of-words is competitive — Cora's features")
    print("  are already informative. The GCN advantage grows on sparser features.")

print(
)

# Register the GCN model
if has_registry:
    version = register_model(
        registry=registry,
        name=f"m5_gcn_{dataset_name.lower().replace(' ', '_')}",
        model=gcn,
        metrics=[
            MetricSpec(name="best_val_accuracy", value=best_val),
            MetricSpec(name="best_test_accuracy", value=best_test),
            MetricSpec(name="final_loss", value=gcn_losses[-1]),
            MetricSpec(name="baseline_accuracy", value=baseline_acc),
            MetricSpec(name="graph_improvement", value=improvement),
            MetricSpec(name="hidden_dim", value=float(HIDDEN_DIM)),
            MetricSpec(name="epochs", value=float(EPOCHS)),
        ],
    )
    print(f"  Registered GCN: version={version.version}, val_acc={best_val:.4f}")



### Apply Checkpoint


In [ ]:
assert baseline_acc > 0.0, "Baseline should produce non-zero accuracy"
print("\n--- Apply checkpoint passed --- GCN vs baseline comparison complete\n")



## REFLECTION


GRAPH CONVOLUTIONAL NETWORK (Kipf & Welling, 2017):
  [x] Message passing as matrix multiplication: H' = A_norm @ H @ W
  [x] Fixed aggregation via degree-normalised adjacency (Laplacian)
  [x] Simplest GNN — fast, parallelisable, effective on homogeneous graphs
  [x] Trained on {dataset_name}: {best_val:.1%} val accuracy, {best_test:.1%} test accuracy
  [x] Compared to bag-of-words baseline: +{improvement*100:.1f} percentage points
  [x] Visualised embeddings showing class separation in 2-D PCA

  KEY LIMITATION: GCN uses FIXED aggregation weights (degree-based).
  All neighbours contribute equally regardless of relevance. What if
  some citations are more important than others?

  Next: Exercise 6.2 — Graph Attention Networks (GAT) learn which
  neighbours matter most via attention weights...


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED — GCN")
print("=" * 70)
print(
)

# Clean up

await conn.close()



## DIAGNOSTIC CHECKPOINT — five instruments before Visualise


In [ ]:
# Reference: `kailash_ml.diagnostics` (via `kailash-ml`) — see gold standard
# `solutions/ex_1/01_standard_ae.py` for the full pattern.
from kailash_ml.diagnostics import run_diagnostic_checkpoint


def _diag_loss(m, batch):
    # GCN node classification loss
    # Customise per your exercise's loss shape.
    if isinstance(batch, (tuple, list)):
        x = batch[0]
        y = batch[1] if len(batch) > 1 else None
    else:
        x, y = batch, None
    out = m(x)
    import torch.nn.functional as F
    if y is None:
        return F.mse_loss(out, x)
    return F.cross_entropy(out, y)


print("\n── Diagnostic Report (GCN — Graph Convolutional Network) ──")
try:
    diag, findings = run_diagnostic_checkpoint(
        gcn,
        [(features, labels)],
        _diag_loss,
        title="GCN — Graph Convolutional Network",
        n_batches=8,
        show=False,
    )
except Exception as exc:
    # Diagnostic is pedagogical — never block the exercise on it.
    print(f"[diagnostic skipped: {exc}]")

# ══════ EXPECTED OUTPUT (synthesized reference — full run produces similar pattern) ══════



## DL Diagnostics Report — Prescription Pad


In [ ]:
# [✓] Gradient flow (HEALTHY): RMS range 5.2e-04 to 8.7e-03 across 2 GCN layers.
#     Shallow GCN = no over-smoothing risk yet.
# [✓] Dead neurons  (HEALTHY): 8% inactive — GCN uses ReLU, healthy.
# [✓] Loss trend    (HEALTHY): train loss → 0.23, val accuracy plateauing at ~82%.



## STUDENT INTERPRETATION GUIDE — reading the Prescription Pad:


In [ ]:
#  [BLOOD TEST] Shallow GCN (2 layers) shows no pathologies. BUT
#     slide 5.6 warns: stack 4+ GCN layers and node embeddings
#     become nearly identical (over-smoothing). Watch for this in
#     ex_6/05 architecture comparison — the signature is all nodes'
#     cosine similarity → 1.0 at deep layers.
#     >> Prescription: use skip connections (ResGCN) OR PairNorm OR
#        stick to 2-3 layers. GAT (ex_6/02) also helps because
#        learned attention weights can avoid smoothing.
#
#  [X-RAY] 8% dead ReLU is normal. GCN's linear aggregation
#     followed by ReLU doesn't typically kill channels.
#
#  [STETHOSCOPE] 82% accuracy on Cora is baseline competitive.
#     GAT and GraphSAGE (next exercises) typically push to 83-85%
#     via learned vs fixed aggregation.

